In [10]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             confusion_matrix)

TASK 1: DATASET SELECTION
1. Dataset Name   : Breast Cancer Wisconsin Dataset
2. Dataset Source : Built-in Scikit-Learn (sklearn.datasets.load_breast_cancer)
3. Number of Rows : 569
  & Number of Cols : 30 features + 1 target
4. Target Column  : target (0 = Malignant, 1 = Benign)
5. Problem Type   : Binary Classification


In [11]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print("Shape:", df.shape)
print("Target classes:", data.target_names)
print(df.head())

Shape: (569, 31)
Target classes: ['malignant' 'benign']
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  wors

TASK 2: PROBLEM UNDERSTANDING
1. Problem being solved: Predicting whether a breast tumor is malignant or benign based on measurements taken from a digitized image of a fine needle aspirate (FNA) of a breast mass.
2. Input Features: 30 numerical features describing characteristics of cell nuclei — including radius, texture, perimeter, area, smoothness, compactness, concavity, and more. Each feature has a mean, standard error, and worst value.
3. Target Variable: 0 = Malignant (cancerous) 1 = Benign (non-cancerous)
4. Real World Usefulness: Early and accurate detection of breast cancer significantly improves survival rates. An ML model that assists doctors in classifying tumors reduces diagnostic errors and speeds up the process, especially in areas with limited access to specialist pathologists.

In [12]:
print("Problem: Binary classification of breast tumors")
print("Features:", data.feature_names[:5], "... and 25 more")
print("Target:", dict(enumerate(data.target_names)))

Problem: Binary classification of breast tumors
Features: ['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness'] ... and 25 more
Target: {0: np.str_('malignant'), 1: np.str_('benign')}


In [13]:
# TASK 3: DATA SPLITTING

X = data.data
y = data.target

# using 80:20 split ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Split Ratio Used : 80% Training / 20% Testing")
print("Training samples :", X_train.shape[0])
print("Testing samples  :", X_test.shape[0])

Split Ratio Used : 80% Training / 20% Testing
Training samples : 455
Testing samples  : 114


In [14]:
# TASK 4: PIPELINE CREATION

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

# fit the pipeline on training data
pipeline.fit(X_train, y_train)

print("Pipeline created and trained successfully")
print("Steps:", pipeline.steps)

Pipeline created and trained successfully
Steps: [('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000))]


In [15]:
# TASK 5: HYPERPARAMETER TUNING WITH GRIDSEARCHCV

param_grid = {
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__solver': ['lbfgs', 'liblinear']
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='accuracy',
    verbose=1
)

grid_search.fit(X_train, y_train)

print("\nHyperparameters Tested:")
print("  C values  :", [0.01, 0.1, 1, 10, 100])
print("  Solvers   :", ['lbfgs', 'liblinear'])
print("\nBest Parameters :", grid_search.best_params_)
print("Best CV Score   :", round(grid_search.best_score_, 4))

Fitting 5 folds for each of 10 candidates, totalling 50 fits

Hyperparameters Tested:
  C values  : [0.01, 0.1, 1, 10, 100]
  Solvers   : ['lbfgs', 'liblinear']

Best Parameters : {'model__C': 0.1, 'model__solver': 'liblinear'}
Best CV Score   : 0.978


In [16]:
# TASK 6: MODEL EVALUATION

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

# calculate all metrics
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
cm        = confusion_matrix(y_test, y_pred)

print("=" * 45)
print("         ML MODEL EVALUATION RESULTS")
print("=" * 45)
print(f"Accuracy  : {round(accuracy, 4)}")
print(f"Precision : {round(precision, 4)}")
print(f"Recall    : {round(recall, 4)}")
print(f"F1 Score  : {round(f1, 4)}")
print("\nConfusion Matrix:")
print(cm)
print("\nMatrix Breakdown:")
print(f"  True Negatives  (Malignant correctly identified) : {cm[0][0]}")
print(f"  False Positives (Malignant wrongly called Benign): {cm[0][1]}")
print(f"  False Negatives (Benign wrongly called Malignant): {cm[1][0]}")
print(f"  True Positives  (Benign correctly identified)    : {cm[1][1]}")

         ML MODEL EVALUATION RESULTS
Accuracy  : 0.9912
Precision : 0.9861
Recall    : 1.0
F1 Score  : 0.993

Confusion Matrix:
[[42  1]
 [ 0 71]]

Matrix Breakdown:
  True Negatives  (Malignant correctly identified) : 42
  False Positives (Malignant wrongly called Benign): 1
  False Negatives (Benign wrongly called Malignant): 0
  True Positives  (Benign correctly identified)    : 71


In [17]:

# TASK 7: RULE-BASED BENCHMARK

# worst radius is column index 20 in the dataset
# medical studies show larger radius strongly
# indicates malignancy in breast tumors

# find the feature index
feature_names = list(data.feature_names)
worst_radius_index = feature_names.index('worst radius')

# findd the best threshold using training data
thresholds = np.percentile(X_train[:, worst_radius_index],
                            np.arange(10, 90, 5))

best_threshold = None
best_rule_acc  = 0

for threshold in thresholds:
    # rule: if worst radius <= threshold → benign (1)
    #       if worst radius >  threshold → malignant (0)
    rule_preds = np.where(
        X_train[:, worst_radius_index] <= threshold, 1, 0
    )
    acc = accuracy_score(y_train, rule_preds)
    if acc > best_rule_acc:
        best_rule_acc  = acc
        best_threshold = threshold


rule_predictions = np.where(
    X_test[:, worst_radius_index] <= best_threshold, 1, 0
)


rule_accuracy  = accuracy_score(y_test, rule_predictions)
rule_precision = precision_score(y_test, rule_predictions)
rule_recall    = recall_score(y_test, rule_predictions)
rule_f1        = f1_score(y_test, rule_predictions)

print("=" * 45)
print("       RULE-BASED BENCHMARK RESULTS")
print("=" * 45)
print(f"Rule Used : If worst radius <= {round(best_threshold, 2)} → Benign")
print(f"            If worst radius >  {round(best_threshold, 2)} → Malignant")
print(f"\nAccuracy  : {round(rule_accuracy, 4)}")
print(f"Precision : {round(rule_precision, 4)}")
print(f"Recall    : {round(rule_recall, 4)}")
print(f"F1 Score  : {round(rule_f1, 4)}")


print("\n" + "=" * 45)
print("         FINAL COMPARISON")
print("=" * 45)
print(f"{'Metric':<15} {'ML Model':>10} {'Rule-Based':>12}")
print("-" * 45)
print(f"{'Accuracy':<15} {round(accuracy, 4):>10} {round(rule_accuracy, 4):>12}")
print(f"{'Precision':<15} {round(precision, 4):>10} {round(rule_precision, 4):>12}")
print(f"{'Recall':<15} {round(recall, 4):>10} {round(rule_recall, 4):>12}")
print(f"{'F1 Score':<15} {round(f1, 4):>10} {round(rule_f1, 4):>12}")
print("=" * 45)

       RULE-BASED BENCHMARK RESULTS
Rule Used : If worst radius <= 16.46 → Benign
            If worst radius >  16.46 → Malignant

Accuracy  : 0.9474
Precision : 0.9333
Recall    : 0.9859
F1 Score  : 0.9589

         FINAL COMPARISON
Metric            ML Model   Rule-Based
---------------------------------------------
Accuracy            0.9912       0.9474
Precision           0.9861       0.9333
Recall                 1.0       0.9859
F1 Score             0.993       0.9589
